# aptdata Quickstart

Welcome to the interactive quickstart for `aptdata`! 

`aptdata` is a declarative, extensible framework for building smart data pipelines in Python. Here you can run some basic examples right in your browser.

First, let's install the library.

In [ ]:
!pip install aptdata[all]

## 1. Creating a Component

Let's create a very simple component that doubles the input data. `aptdata` provides the `BaseComponent` class which we will extend.

In [ ]:
from pydantic.dataclasses import dataclass as pydantic_dataclass

from aptdata.core import BaseComponent, BaseDataset, IDataset


@pydantic_dataclass
class MemoryDataset(BaseDataset):
    def __post_init__(self): self._data = None
    def read(self): return self._data
    def write(self, data): self._data = data

@pydantic_dataclass
class DoubleComponent(BaseComponent):
    def validate_inputs(self, inputs: list[IDataset]) -> bool:
        return len(inputs) == 1

    def execute(self, inputs: list[IDataset]) -> list[IDataset]:
        out = MemoryDataset(uri="memory://out")
        out.write([x * 2 for x in inputs[0].read()])
        return [out]

print("Component successfully defined!")

## 2. Running the Component

Now we can create some data and pass it to our component to see the execution in action.

In [ ]:
input_dataset = MemoryDataset(uri="memory://in")
input_dataset.write([1, 2, 3, 4, 5])

component = DoubleComponent("my_doubler")
output_datasets = component.execute([input_dataset])

print("Input:", input_dataset.read())
print("Output:", output_datasets[0].read())

## 3. Data Quality Validation

Let's look at one of the out-of-the-box features: Data Quality Expectations.

In [ ]:
import pandas as pd

from aptdata.plugins.quality import (
    EnforcementMode,
    ExpectColumnToNotBeNull,
    QualityValidator,
)

validator = QualityValidator(
    expectations=[ExpectColumnToNotBeNull("id")],
    enforcement=EnforcementMode.ABORT,
)

# Create data with a missing ID
raw_df = pd.DataFrame({"id": [1, 2, None], "value": ["A", "B", "C"]})

try:
    clean_data = validator.validate(raw_df)
except Exception as e:
    print(f"Validation naturally failed: {e}")